features aus Mediapipe-Landmarks bauen 

---

In [1]:
# Imports
#  
import numpy as np
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from scipy.signal import resample
import sqlite3
import pandas as pd
import sqlite3
import pandas as pd
import duckdb
import polars as pl


In [3]:

# if the notebook is in the same folder as the db file:
con = duckdb.connect("landmark_database.db")

# otherwise:
# con = duckdb.connect(r"C:\Users\lejza\spice_bootcamp\landmark_database.db")


In [4]:
#  Load data as csv from MediaPipe Pose from database

#Connect to database
db_path =r"C:\Users\lejaz\spice_bootcamp\landmark_database.db"
conn = sqlite3.connect(db_path)

#==========================================
#BASIC QUERIES
#==========================================
#View all data (limit to first 1000 rows)
df = pd.read_sql_query("SELECT * FROM landmarks LIMIT 100000", conn)
print(f"Total rows loaded: {len(df)}")
df.head()

Test_data = df.copy()

#df.head()

#Rows: 21,850,000 - 22,050,000


Total rows loaded: 100000


In [5]:
#View all data (limit to first 1000 rows)
df = pd.read_sql_query("SELECT * FROM landmarks LIMIT 100000", conn)
print(f"Total rows loaded: {len(df)}")
df.head()

Total rows loaded: 100000


,id,patient_name,frame,movement_type,jacket_status,side,model_name,timestamp_ms,landmark_id,x_norm,...,title,uploader,fps,start_time,end_time,duration,checksum,width,height,created_at
0,1,PA000,0,Fast Movement,With Jacket,Right,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:38
1,2,PA000,1,Fast Movement,With Jacket,Right,DensePose,33.0,0,NaN,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:38
2,3,PA000,2,Fast Movement,With Jacket,Right,DensePose,66.0,0,NaN,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:38
3,4,PA000,3,Fast Movement,With Jacket,Right,DensePose,100.0,0,NaN,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:38
4,5,PA000,4,Fast Movement,With Jacket,Right,DensePose,133.0,0,NaN,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:38


In [6]:
Test_data.tail(30)

,id,patient_name,frame,movement_type,jacket_status,side,model_name,timestamp_ms,landmark_id,x_norm,...,title,uploader,fps,start_time,end_time,duration,checksum,width,height,created_at
99970,99971,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,21,0.711586,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99971,99972,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,22,0.725127,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99972,99973,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,23,0.696310,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99973,99974,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,24,0.701167,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99974,99975,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,25,0.707770,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99975,99976,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,26,0.704635,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99976,99977,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,27,0.666555,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99977,99978,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,28,0.692548,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99978,99979,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,29,0.651192,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99979,99980,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,30,0.683132,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41


---

keep gait_preprocessing_pipeline.py as authoritative module

In [22]:
import gait_preprocessing_pipeline as gait


- turn the long-format MediaPipe data into a df_video
    - one row per video with a pose tensor

- take one normalized gait clip, e.g. (T,33,3), and return a dict of scalar gait features

- df_video als preprocess into clips via the given pipelien and than with that a feature dataframe

In [39]:
%load_ext autoreload
%autoreload 2

import feature_extraction as fx
import gait_preprocessing_pipeline as gait
import numpy as np
import pandas as pd


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
%reload_ext autoreload
%autoreload 2
import importlib
importlib.reload(fx)


<module 'feature_extraction' from 'c:\\Users\\lejaz\\spice_bootcamp\\GAITy-Capstone-Modeling\\notebooks\\feature_extraction.py'>

In [44]:
dummy_clip = np.zeros((60, gait.N_JOINTS, 3), dtype=np.float32)
fx.compute_clip_features(dummy_clip)

{'step_height_L': 0.0,
 'step_height_R': 0.0,
 'step_length_L': 0.0,
 'step_length_R': 0.0,
 'pelvis_drop_mean': 0.0,
 'pelvis_drop_std': 0.0,
 'trunk_lean_mean': 0.0,
 'trunk_lean_std': 0.0,
 'heel_range_L': 0.0,
 'heel_range_R': 0.0,
 'step_height_symmetry': 0.0,
 'step_length_symmetry': 0.0}

In [47]:
whos


Variable            Type                  Data/Info
---------------------------------------------------
LEFT_HEEL           int                   29
LEFT_HIP            int                   23
LEFT_SHOULDER       int                   11
RIGHT_HEEL          int                   30
RIGHT_HIP           int                   24
RIGHT_SHOULDER      int                   12
Test_data           DataFrame             Shape: (100000, 35)
add_pose_column     function              <function add_pose_column at 0x000001EE90A25440>
con                 DuckDBPyConnection    <_duckdb.DuckDBPyConnecti<...>ct at 0x000001EE8A462AF0>
conn                Connection            <sqlite3.Connection object at 0x000001EEE949FB50>
db_path             str                   C:\Users\lejaz\spice_boot<...>camp\landmark_database.db
df                  DataFrame             Shape: (100000, 35)
duckdb              module                <module 'duckdb' from 'c:<...>es\\duckdb\\__init__.py'>
dummy_clip          ndarr

In [48]:
from gait_preprocessing_pipeline import add_pose_column

df_video = add_pose_column(Test_data)
df_video.head()


,id,patient_name,frame,movement_type,jacket_status,side,model_name,timestamp_ms,landmark_id,x_norm,...,uploader,fps,start_time,end_time,duration,checksum,width,height,created_at,pose
0,1,PA000,0,Fast Movement,With Jacket,Right,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,2026-01-14 16:47:38,"[[[nan, nan, nan], [0.0, 0.0, 0.0], [0.0, 0.0,..."
5504,5505,PA000,0,Fast Movement,With Jacket,Left,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,2026-01-14 16:47:38,"[[[nan, nan, nan], [0.0, 0.0, 0.0], [0.0, 0.0,..."
10902,10903,PA000,0,Fast Movement,Without Jacket,Right,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,2026-01-14 16:47:39,"[[[nan, nan, nan], [0.0, 0.0, 0.0], [0.0, 0.0,..."
15616,15617,PA000,0,Fast Movement,Without Jacket,Left,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,2026-01-14 16:47:39,"[[[nan, nan, nan], [0.0, 0.0, 0.0], [0.0, 0.0,..."
21008,21009,PA000,0,Regular Movement,With Jacket,Right,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,2026-01-14 16:47:39,"[[[nan, nan, nan], [0.0, 0.0, 0.0], [0.0, 0.0,..."


In [49]:
df_video.iloc[0]["pose"].shape


(192, 33, 3)

video level dataframe is ready

---

In [50]:
df_video.columns


Index(['id', 'patient_name', 'frame', 'movement_type', 'jacket_status', 'side',
       'model_name', 'timestamp_ms', 'landmark_id', 'x_norm', 'y_norm',
       'z_norm', 'visibility', 'x_px', 'y_px', 'source_file', 'file_order',
       'file_path', 'start_frame', 'end_frame', 'url', 'gait_event', 'dataset',
       'gait_pattern', 'add_pattern_info', 'title', 'uploader', 'fps',
       'start_time', 'end_time', 'duration', 'checksum', 'width', 'height',
       'created_at', 'pose'],
      dtype='object')

In [51]:
# 1) check pose shape
df_video.iloc[0]["pose"].shape

# 2) check what dataset labels look like
df_video["dataset"].value_counts()

# 3) rough fps overview
df_video["fps"].describe()


count       0
unique      0
top       NaN
freq      NaN
Name: fps, dtype: object

In [52]:
label_map_for_strings = {
    "normal": "normal gait",
    "abnormal": "abnormal gait",
    "healthy": "normal gait",
    "pathological": "abnormal gait",
    # add whatever you actually have
}

df_video["dataset"] = df_video["dataset"].map(label_map_for_strings)




from xgboost import XGBClassifier

X = df_features.drop(columns=["label"])
y = df_features["label"]

model = XGBClassifier()
model.fit(X, y)


---

now the actual feature calculation! :) 

---

In [53]:
df_video["movement_type"].value_counts()


movement_type
Fast Movement       8
Regular Movement    8
Name: count, dtype: int64

In [54]:
df_video["side"].value_counts()


side
Right    8
Left     8
Name: count, dtype: int64

In [55]:
df_video["gait_event"].value_counts()


Series([], Name: count, dtype: int64)

---

## set up generic base functions for any landmarker (set) to calculate the functions
joint_speed => underlying speed signal
moving_and_still times => how long moves/not moves
range_of_motion => ROM for a joint

In [72]:
%load_ext autoreload
%autoreload 2

import feature_extraction as fx
import gait_preprocessing_pipeline as gait
import numpy as np

dummy_clip = np.zeros((60, gait.N_JOINTS, 3), dtype=np.float32)
fx.compute_clip_features(dummy_clip, fps=30.0)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


{'step_height_L': 0.0,
 'step_height_R': 0.0,
 'step_length_L': 0.0,
 'step_length_R': 0.0,
 'pelvis_drop_mean': 0.0,
 'pelvis_drop_std': 0.0,
 'trunk_lean_mean': 0.0,
 'trunk_lean_std': 0.0,
 'heel_range_L': 0.0,
 'heel_range_R': 0.0,
 'step_height_symmetry': 0.0,
 'step_length_symmetry': 0.0,
 'knee_L_moving_time_sec': 0.0,
 'knee_L_still_time_sec': 1.9666666666666666,
 'knee_L_moving_fraction': 0.0,
 'knee_L_still_fraction': 1.0,
 'knee_L_mean_speed': 0.0,
 'knee_L_max_speed': 0.0,
 'knee_L_total_time_sec': 1.9666666666666666,
 'knee_R_moving_time_sec': 0.0,
 'knee_R_still_time_sec': 1.9666666666666666,
 'knee_R_moving_fraction': 0.0,
 'knee_R_still_fraction': 1.0,
 'knee_R_mean_speed': 0.0,
 'knee_R_max_speed': 0.0,
 'knee_R_total_time_sec': 1.9666666666666666,
 'knee_L_rom_y': 0.0,
 'knee_R_rom_y': 0.0}

In [73]:
[k for k in fx.compute_clip_features(dummy_clip, 30.0).keys() if "knee" in k]


['knee_L_moving_time_sec',
 'knee_L_still_time_sec',
 'knee_L_moving_fraction',
 'knee_L_still_fraction',
 'knee_L_mean_speed',
 'knee_L_max_speed',
 'knee_L_total_time_sec',
 'knee_R_moving_time_sec',
 'knee_R_still_time_sec',
 'knee_R_moving_fraction',
 'knee_R_still_fraction',
 'knee_R_mean_speed',
 'knee_R_max_speed',
 'knee_R_total_time_sec',
 'knee_L_rom_y',
 'knee_R_rom_y']

In [74]:
['knee_L_moving_time_sec', 'knee_L_still_time_sec', ..., 'knee_L_rom_y',
 'knee_R_moving_time_sec', ..., 'knee_R_rom_y']


['knee_L_moving_time_sec',
 'knee_L_still_time_sec',
 Ellipsis,
 'knee_L_rom_y',
 'knee_R_moving_time_sec',
 Ellipsis,
 'knee_R_rom_y']

In [75]:
df_features = fx.extract_features_from_df_video(df_video)
df_features.head()


,step_height_L,step_height_R,step_length_L,step_length_R,pelvis_drop_mean,pelvis_drop_std,trunk_lean_mean,trunk_lean_std,heel_range_L,heel_range_R,...,knee_R_max_speed,knee_R_total_time_sec,knee_L_rom_y,knee_R_rom_y,label_fine,label_class,label_id,movement_type,side,source_file
0,1.726805,1.695221,1.346603,1.138875,0.020217,0.024436,-0.030305,0.147717,1.919934,1.857972,...,10.283575,6.366667,1.010534,1.041594,None,None,None,Fast Movement,Right,semantic_segmentation_PA000_FGS_WJ_1_DensePose...
1,2.526946,3.045800,1.524194,1.609756,0.015201,0.083514,-0.054901,0.113512,2.644770,3.116001,...,20.468031,6.033333,1.380294,1.586991,None,None,None,Fast Movement,Left,semantic_segmentation_PA000_FGS_WJ_2_DensePose...
2,1.819934,1.783279,1.225501,1.201241,0.011532,0.021964,0.008385,0.158382,1.991898,1.952520,...,11.219465,5.633333,0.969453,0.905977,None,None,None,Fast Movement,Right,semantic_segmentation_PA000_FGS_WoJ_1_DensePos...
3,2.067794,2.718333,1.551333,1.546484,0.001384,0.066918,-0.039065,0.135969,2.124789,2.972578,...,12.060044,5.833333,1.126693,1.253214,None,None,None,Fast Movement,Left,semantic_segmentation_PA000_FGS_WoJ_2_DensePos...
4,1.810680,1.735085,1.193693,1.035499,0.018124,0.019447,-0.041142,0.152405,1.958019,1.903667,...,12.002893,6.700000,0.963531,0.848466,None,None,None,Regular Movement,Right,semantic_segmentation_PA000_UGS_WJ_1_DensePose...


In [76]:
df_features.head()
df_features.columns


Index(['step_height_L', 'step_height_R', 'step_length_L', 'step_length_R',
       'pelvis_drop_mean', 'pelvis_drop_std', 'trunk_lean_mean',
       'trunk_lean_std', 'heel_range_L', 'heel_range_R',
       'step_height_symmetry', 'step_length_symmetry',
       'knee_L_moving_time_sec', 'knee_L_still_time_sec',
       'knee_L_moving_fraction', 'knee_L_still_fraction', 'knee_L_mean_speed',
       'knee_L_max_speed', 'knee_L_total_time_sec', 'knee_R_moving_time_sec',
       'knee_R_still_time_sec', 'knee_R_moving_fraction',
       'knee_R_still_fraction', 'knee_R_mean_speed', 'knee_R_max_speed',
       'knee_R_total_time_sec', 'knee_L_rom_y', 'knee_R_rom_y', 'label_fine',
       'label_class', 'label_id', 'movement_type', 'side', 'source_file'],
      dtype='object')

- this results in completely numerical feature columns
- labels: label_fine, label_class, label_id
- meta: movement_type, side, source_file 

Based on this, a multi class basmodel can be build. 